# Vidu Q2 Turbo Image-to-Video API 使用指南

本 Notebook 演示如何通过 **七牛 AIToken** 平台调用 **Vidu Q2 Turbo (Image-to-Video)** 模型 API。

## 模型简介

Vidu Q2 Turbo Image-to-Video 是一款**高速**首尾帧生成视频的模型，API 参数与 Q2 Pro 完全一致，但生成速度更快，适合对时效性要求较高的场景。

- **首尾帧驱动**：提供起始和结束画面，模型自动补全中间过渡
- **速度更快**：相比 Pro 版本，Turbo 版本生成速度显著提升
- **提示词辅助**：可通过 prompt 描述过渡方式和场景细节
- **推荐提示词**：支持 `is_rec: true` 由系统自动推荐提示词
- **运动幅度控制**：支持 auto、small、medium、large 四种运动幅度

> **Turbo 与 Pro 的区别**：两者 API 参数完全相同，Turbo 版本生成速度更快，适合需要快速出片的场景。

**API 端点**：`fal-ai/vidu/q2/image-to-video/turbo`

## 1. 环境准备

首先安装 `fal-client` 依赖包，并设置 API Key 和七牛 AIToken 队列服务地址。

In [ ]:
# 安装 fal-client
%pip install fal-client -q

In [ ]:
import os

# 设置七牛 AIToken 队列服务地址
os.environ["FAL_QUEUE_RUN_HOST"] = "api.qnaigc.com/queue"

# 设置 FAL_KEY 环境变量
# 你也可以在终端中通过 export FAL_KEY="xxx" 来设置
os.environ["FAL_KEY"] = os.environ.get("QINIU_API_KEY") or (_ for _ in ()).throw(ValueError("环境变量 QINIU_API_KEY 未设置"))

In [ ]:
import fal_client
from IPython.display import Video, display

## 2. 基础用法 — 队列调用

七牛 AIToken 平台使用队列模式调用 API：先提交请求（`submit`），然后轮询状态（`status`），最后获取结果（`result`）。

下面先封装一个通用的调用辅助函数，后续示例都会复用。

In [ ]:
import time

def run_fal_task(endpoint, arguments, poll_interval=5):
    """提交任务并轮询等待结果返回"""
    # 步骤 1：提交请求
    handler = fal_client.submit(endpoint, arguments=arguments)
    request_id = handler.request_id
    print(f"请求已提交，request_id: {request_id}")

    # 步骤 2：轮询任务状态
    while True:
        status = fal_client.status(endpoint, request_id, with_logs=True)
        print(f"当前状态: {type(status).__name__}")

        if isinstance(status, fal_client.Completed):
            print("任务完成!")
            break

        if hasattr(status, "logs") and status.logs:
            for log in status.logs:
                print(log["message"])

        time.sleep(poll_interval)

    # 步骤 3：获取结果
    return fal_client.result(endpoint, request_id)


# 基础调用示例：首帧 + 尾帧 + prompt
result = run_fal_task(
    "fal-ai/vidu/q2/image-to-video/turbo",
    arguments={
        "prompt": "Dragon lands on a rock",
        "image_url": "https://v3.fal.media/files/zebra/sgsdKvPigPhJ1S7Hl5bWc_first_frame_q1.png",
        "end_image_url": "https://v3.fal.media/files/kangaroo/CASBu_OmOnZ8IafirarFL_last_frame_q1.png",
    },
)

# 打印返回结果
print(result)

In [ ]:
# 展示生成的视频
video_url = result["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 3. 参数说明与高级用法

### 可配置参数一览

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `image_url` | string | *必填* | 首帧图片 URL 或 Base64 编码 |
| `end_image_url` | string | *必填* | 尾帧图片 URL 或 Base64 编码 |
| `prompt` | string | 可选 | 文本提示词，描述过渡方式，最多 2000 字符 |
| `seed` | integer | 随机 | 随机种子，默认不传或传 0 使用随机数 |
| `duration` | integer | `5` | 视频时长（秒） |
| `resolution` | string | `"1080p"` | 分辨率，可选：`1080p` |
| `movement_amplitude` | string | `"auto"` | 运动幅度：`auto`、`small`、`medium`、`large` |

> **图片要求**：支持 png、jpeg、jpg、webp 格式，比例小于 1:4 或 4:1，大小不超过 50MB。


## 4. 队列模式 — 手动分步调用

上面的 `run_fal_task` 封装了完整的队列流程。如果你需要更精细的控制（例如在提交后做其他事情，稍后再取结果），可以手动分步调用。

In [ ]:
# 步骤 1：提交请求
handler = fal_client.submit(
    "fal-ai/vidu/q2/image-to-video/turbo",
    arguments={
        "prompt": "A smooth cinematic transition from day to night",
        "image_url": "https://v3.fal.media/files/zebra/sgsdKvPigPhJ1S7Hl5bWc_first_frame_q1.png",
        "end_image_url": "https://v3.fal.media/files/kangaroo/CASBu_OmOnZ8IafirarFL_last_frame_q1.png",
    },
)

# 获取请求 ID，用于后续查询
request_id = handler.request_id
print(f"请求已提交，request_id: {request_id}")

In [ ]:
# 步骤 2：查询任务状态
import time

while True:
    status = fal_client.status(
        "fal-ai/vidu/q2/image-to-video/turbo",
        request_id,
        with_logs=True,
    )
    print(f"当前状态: {type(status).__name__}")

    if isinstance(status, fal_client.Completed):
        print("任务完成!")
        break

    # 每 5 秒查询一次
    time.sleep(5)

In [ ]:
# 步骤 3：获取结果
result_async = fal_client.result(
    "fal-ai/vidu/q2/image-to-video/turbo",
    request_id,
)

# 展示生成的视频
video_url = result_async["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 5. Schema 参考

### Input Schema

```json
{
  "image_url": "string (必填, 首帧图片 URL 或 Base64)",
  "end_image_url": "string (必填, 尾帧图片 URL 或 Base64)",
  "prompt": "string (可选, 最多 2000 字符)",
  "is_rec": "boolean (可选, 默认 false, 是否使用推荐提示词)",
  "seed": "integer (可选, 默认随机)",
  "duration": "integer (可选, 默认 5)",
  "resolution": "1080p (可选, 默认 1080p)",
  "movement_amplitude": "auto | small | medium | large (可选, 默认 auto)"
}
```

### Output Schema (队列提交响应)

```json
{
  "status": "IN_QUEUE",
  "request_id": "string (任务唯一 ID)",
  "response_url": "string (查询任务结果的 URL)",
  "status_url": "string (查询任务状态的 URL)",
  "cancel_url": "string (取消任务的 URL, 暂未支持)"
}
```

### 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [fal-client Python 文档](https://docs.fal.ai/reference/client-libraries/python/fal_client)
- [七牛 API 调用示例文档](https://apidocs.qnaigc.com)